# Wild Rift Player Churn Prediction

This notebook compares Random Forest, XGBoost, and LightGBM for player churn prediction, with a focus on early-session risk, cold-start behavior, and deployment-oriented evaluation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)

## 1. Problem Statement

Player churn is a critical business problem in mobile gaming because many users disengage within their first few sessions. In this setting, missing a true churner is often more costly than generating a moderate number of false positives. 

This notebook evaluates three machine learning approaches for churn prediction:
- Random Forest
- XGBoost
- LightGBM

The comparison focuses on:
- AUC
- precision
- recall
- F1-score
- practical suitability for deployment

## 2. Load Data

Load the dataset and inspect the target column, missing values, and feature types.

In [ ]:
# change filename to your actual file
file_path = "predict_online_gaming_behavior.csv"

df = pd.read_csv(file_path)
df.head()

In [ ]:
df.shape, df.columns.tolist()[:20]

In [ ]:
# set your target column here
TARGET = "churn"

print(df[TARGET].value_counts(dropna=False))
print(df[TARGET].value_counts(normalize=True, dropna=False))

## 3. Data Overview

This section checks missing values, column types, and a few high-level characteristics of the dataset.

In [ ]:
df.info()

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0].head(20)

In [ ]:
categorical_cols = df.drop(columns=[TARGET]).select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = df.drop(columns=[TARGET]).select_dtypes(include=["number"]).columns.tolist()

print("Categorical columns:", len(categorical_cols))
print("Numeric columns:", len(numeric_cols))

## 4. Prepare Features

Split the dataset into features and target, then define preprocessing steps for numeric and categorical variables.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

## 5. Define Models

Three models are compared using the same preprocessing pipeline and the same cross-validation setup.

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        random_state=42,
        class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
        class_weight="balanced"
    )
}

## 6. Cross-Validation Model Comparison

This section compares the three models using stratified 10-fold cross-validation.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "auc": "roc_auc"
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "AUC": scores["test_auc"].mean()
    })

results_df = pd.DataFrame(results).sort_values("Recall", ascending=False)
results_df.round(3)

## 7. Model Performance Visualization

Plot model performance across key metrics.

In [ ]:
plot_df = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
plt.ylim(0.6, 1.0)
plt.title("Model Performance Comparison Across Key Metrics")
plt.tight_layout()
plt.show()

## 8. Final LightGBM Evaluation

Because LightGBM provides the strongest recall and good practical performance, it is selected for deeper inspection.

In [ ]:
lightgbm_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", models["LightGBM"])
])

pred = cross_val_predict(lightgbm_pipe, X, y, cv=cv, method="predict")
proba = cross_val_predict(lightgbm_pipe, X, y, cv=cv, method="predict_proba")[:, 1]

metrics_summary = {
    "Accuracy": accuracy_score(y, pred),
    "Precision": precision_score(y, pred),
    "Recall": recall_score(y, pred),
    "F1": f1_score(y, pred),
    "AUC": roc_auc_score(y, proba)
}

pd.Series(metrics_summary).round(3)

In [ ]:
cm = confusion_matrix(y, pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Stay", "Churn"], yticklabels=["Stay", "Churn"])
plt.title("Confusion Matrix for LightGBM Churn Prediction")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 9. Retention vs Churn by Session Count

This view helps illustrate how churn is concentrated in the earliest sessions.

In [ ]:
# replace with your real column name
session_col = "session_count"

tmp = df[[session_col, TARGET]].copy()
tmp[session_col] = tmp[session_col].clip(upper=7)

summary = (
    tmp.groupby([session_col, TARGET])
       .size()
       .reset_index(name="count")
)
summary["pct"] = summary.groupby(session_col)["count"].transform(lambda s: 100 * s / s.sum())

pivot = summary.pivot(index=session_col, columns=TARGET, values="pct").fillna(0)

if 0 in pivot.columns and 1 in pivot.columns:
    pivot = pivot.rename(columns={0: "Stay", 1: "Churn"})

pivot[[col for col in ["Churn", "Stay"] if col in pivot.columns]].plot(
    kind="bar", figsize=(9, 5), color=["orange", "steelblue"]
)
plt.title("Player Retention vs. Churn by Session Count")
plt.xlabel("Session Count")
plt.ylabel("Percentage of Players")
plt.tight_layout()
plt.show()

## 10. SHAP Explainability

Use SHAP to understand which features contribute most to churn risk predictions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

lightgbm_pipe.fit(X_train, y_train)

X_test_processed = lightgbm_pipe.named_steps["prep"].transform(X_test)
lightgbm_model = lightgbm_pipe.named_steps["model"]

explainer = shap.TreeExplainer(lightgbm_model)
shap_values = explainer.shap_values(X_test_processed)

In [ ]:
shap.summary_plot(shap_values, X_test_processed, show=False)
plt.tight_layout()
plt.show()

## 11. Lifecycle Risk Heatmap

This is a project-level visualization to summarize the relative importance of different data and deployment risks across the ML lifecycle.

In [ ]:
risk_df = pd.DataFrame({
    "Collection": [2, 3, 2, 0, 3],
    "Preprocessing": [2, 2, 3, 0, 2],
    "Model Training": [1, 3, 1, 2, 3],
    "Inference": [1, 2, 1, 3, 2],
    "Deployment": [1, 2, 1, 3, 3]
}, index=["Missing Data", "Bias", "Time Inconsistency", "Explainability Risk", "Cold Start"])

plt.figure(figsize=(8, 5))
sns.heatmap(risk_df, annot=True, cmap="YlOrRd", cbar=True)
plt.title("Data Risk Heatmap Across Lifecycle Stages")
plt.tight_layout()
plt.show()

## 12. Key Findings

Main conclusions from the analysis:

- XGBoost produced the strongest AUC and precision.
- LightGBM produced the strongest recall, which is more important for minimizing missed churners.
- Random Forest is useful as an interpretable baseline but weaker for production deployment in this use case.
- Cold-start sparsity, telemetry quality, and regional imbalance remain important risks beyond model choice alone.
- LightGBM is the most practical deployment choice when balancing recall, speed, and operational fit.